In [1]:
import torch
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from utils.io_utils import prepare_input, derive_step_rewards, prepare_batch_input_for_model

In [2]:
model_name = "RLHFlow/Llama3.1-8B-PRM-Deepseek-Data"

In [3]:
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", torch_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [4]:
df = pd.read_parquet("data/processbench.parquet").sample(100)
questions = df["problem"].values
steps = df["steps"].values

In [5]:
all_rewards = []

input_ids_all = []
token_mask_all = []
for (question, step) in tqdm(zip(questions, steps), total=len(questions), desc="[PRM] Preparing input"):
    input_ids, token_mask = prepare_input(
                            model_name, 
                            problem=question, 
                            steps=step, 
                            tokenizer=tokenizer,
                            convert_to_list=False
    )
    input_ids_all.append(input_ids)
    token_mask_all.append(token_mask)

[PRM] Preparing input:   0%|          | 0/100 [00:00<?, ?it/s]

In [6]:
num_samples = len(questions)
batch_size = 8
for start_idx in tqdm(range(0, num_samples, batch_size), desc="[PRM] Scoring"):
    end_idx = start_idx + batch_size

    batch_input_ids = input_ids_all[start_idx:end_idx]
    batch_token_masks = token_mask_all[start_idx:end_idx]

    batch_input_ids, batch_token_masks = prepare_batch_input_for_model(batch_input_ids, batch_token_masks, pad_token_id=0)

    with torch.no_grad():
        batch_logits_cpu = model(batch_input_ids).logits.cpu()

    rewards = derive_step_rewards(
        model_name,
        batch_logits_cpu,
        batch_token_masks.cpu(),
        tokenizer
    )

    all_rewards.extend(rewards)

[PRM] Scoring:   0%|          | 0/13 [00:00<?, ?it/s]

In [7]:
df[f"{model_name}--rewards"] = all_rewards

In [8]:
df

,id,generator,problem,steps,final_answer_correct,label,split,steps_len,per_step_len,Qwen2.5-Math-PRM-7B,Skywork-o1-Open-PRM-Qwen-2.5-7B,RLHFlow/Llama3.1-8B-PRM-Deepseek-Data--rewards
1237,math-837,Qwen2-7B-Instruct,Find the range of the function $f(x) = \arctan...,[To find the range of the function \(f(x) = \a...,True,3,math,7,"[156, 213, 246, 529, 327, 276, 244]","[1.0, 0.97265625, 0.99609375, 0.2431640625, 0....","[0.28457600421652673, 0.2966542546789911, 0.27...","[0.0330810546875, 0.19677734375, 0.32763671875..."
2417,omnimath-17,Qwen2.5-72B-Instruct,How many regions of the plane are bounded by t...,[To determine how many regions of the plane ar...,False,2,omnimath,9,"[215, 100, 221, 718, 164, 233, 227, 481, 181]","[0.9921875, 0.9921875, 0.9765625, 0.9375, 0.96...","[0.05665242530797383, 0.06097518086496481, 0.0...","[0.615234375, 0.7431640625, 0.822265625, 0.817..."
566,math-166,Qwen2-1.5B-Instruct,Let $P(x)$ be a polynomial such that\n\[P(P(x)...,[Let $P(x) = ax^2 + bx + c$. Then $P(P(x)) = a...,False,0,math,8,"[154, 172, 348, 145, 173, 112, 117, 83]","[0.1962890625, 0.9765625, 0.0211181640625, 0.7...","[0.1789558876900955, 0.23510838921999314, 0.14...","[0.2421875, 0.48828125, 0.47265625, 0.57373046..."
3321,omnimath-921,Qwen2-7B-Instruct,"Find the number of pairs $(a, b)$ of positive ...","[To solve this problem, we need to understand ...",True,0,omnimath,6,"[604, 184, 507, 593, 277, 101]","[0.1298828125, 0.90625, 0.94140625, 0.12402343...","[0.1233656243727556, 0.13939637966532398, 0.12...","[0.03515625, 0.1895751953125, 0.262939453125, ..."
2902,omnimath-502,Llama-3.1-8B-Instruct,Draw a square of side length 1. Connect its si...,[To find the sum of the areas of all the squar...,True,2,omnimath,6,"[210, 354, 396, 257, 404, 91]","[1.0, 0.94921875, 0.57421875, 0.86328125, 0.97...","[0.7371581626286834, 0.8519528019683106, 0.270...","[0.92724609375, 0.93359375, 0.953125, 0.967285..."
...,...,...,...,...,...,...,...,...,...,...,...,...
979,math-579,Qwen2-72B-Instruct,"Let $x,$ $y,$ and $z$ be positive real numbers...",[To find the minimum value of the expression \...,True,6,math,13,"[373, 229, 143, 128, 109, 109, 247, 100, 156, ...","[0.9765625, 0.984375, 0.99609375, 0.8984375, 0...","[0.19193278644723683, 0.40028040867233783, 0.5...","[0.441650390625, 0.64794921875, 0.7490234375, ..."
1736,olympiadbench-336,Llama-3.1-8B-Instruct,"Determine all pairs $(a, b)$ of positive integ...","[To find all pairs \((a, b)\) of positive inte...",False,2,olympiadbench,3,"[526, 297, 280]","[0.9453125, 0.98828125, 0.00897216796875]","[0.48047867804790706, 0.5760650647269683, 0.04...","[0.20947265625, 0.307373046875, 0.28466796875]"
3332,omnimath-932,Qwen2-72B-Instruct,"Find all triples $ (x,y,z)$ of real numbers th...","[To solve this system of equations, we'll try ...",True,5,omnimath,13,"[261, 116, 93, 279, 183, 281, 96, 227, 694, 25...","[0.9609375, 1.0, 1.0, 0.875, 0.9921875, 0.1572...","[0.10521053670308636, 0.09268777863937604, 0.0...","[0.27197265625, 0.63330078125, 0.68603515625, ..."
1619,olympiadbench-219,Qwen2.5-Math-7B-Instruct,"If $1, x, y$ is a geometric sequence and $x, y...","[Given that \(1, x, y\) is a geometric sequenc...",False,3,olympiadbench,10,"[178, 176, 87, 150, 102, 173, 91, 175, 93, 248]","[0.9921875, 1.0, 1.0, 0.9375, 1.0, 1.0, 1.0, 1...","[0.7772998611746911, 0.8824278664544911, 0.903...","[0.4111328125, 0.7431640625, 0.921875, 0.76367..."


----